# 03 · Preparación e ingeniería de datos

**Proyecto:** Identificación de hogares en situación de pobreza monetaria mediante aprendizaje automático (ENAHO 2025).

Este cuaderno corresponde a la **Fase 3 de CRISP-DM: Preparación de los datos**. Aquí se toman y justifican todas las decisiones que transforman el dataset de hogares en la entrada de los modelos, y se demuestra que **no existe fuga de información (data leakage)** entre entrenamiento, validación y prueba.

**Entrada:** el dataset de hogares 2025 del cuaderno 01. **Salida:** la partición en entrenamiento y prueba (`data/processed/particion_2025.csv`) y la marca de los hogares del panel (`data/processed/panel_2025.csv`).

Contenido:

1. resumen de las decisiones de preparación;
2. duplicados;
3. selección de variables;
4. ingeniería de variables (*feature engineering*);
5. división en entrenamiento y prueba;
6. pipeline de preprocesamiento: valores faltantes, codificación y estandarización;
7. valores atípicos y reducción de dimensionalidad;
8. **prevención de data leakage**;
9. guardado de los artefactos para el modelado;
10. hogares vecinos (conglomerados) en la validación cruzada.

> **Regla del proyecto:** ninguna transformación se hace para "mejorar las métricas". Cada decisión se justifica por el tipo de dato o por el uso real que tendría el modelo.

In [1]:
# ==========================================================
# CONFIGURACIÓN DEL ENTORNO (funciona en Google Colab y en local)
# ==========================================================
# En Colab, la primera vez conviene instalar las versiones del proyecto y reiniciar la sesión:
# !pip install -r /content/drive/MyDrive/proyecto-pobreza-ia/requirements.txt
import os
import sys

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # En Colab: montamos Google Drive y nos ubicamos en la carpeta del proyecto
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_PROYECTO = "/content/drive/MyDrive/proyecto-pobreza-ia"  # ajustar si el proyecto está en otra carpeta
    os.chdir(RUTA_PROYECTO)
elif os.path.basename(os.getcwd()) == "notebooks":
    # En local: si el notebook se abre desde la carpeta notebooks/, subimos a la raíz
    os.chdir("..")

# Comprobamos que estamos en la raíz del proyecto y permitimos importar su código (carpeta src/)
assert os.path.isdir("src"), "No se encuentra la carpeta src/: revisa RUTA_PROYECTO"
sys.path.insert(0, os.getcwd())
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Carpeta del proyecto:", os.path.basename(os.getcwd()))

Entorno: local | Carpeta del proyecto: proyecto-pobreza-ia


In [2]:
# Librerías
import pandas as pd
import numpy as np

# Código propio del proyecto (carpeta src/)
from src.config import SEMILLA, PROPORCION_PRUEBA, RUTA_HOGARES, RUTA_PARTICION, RUTA_PANEL_2025
from src.data.cargar import leer_csv_enaho, ruta_modulo
from src.data.descargar import descargar_enaho
from src.features.construir import (
    CLAVES_HOGAR, COLUMNAS_IDENTIFICACION, VARIABLE_OBJETIVO, VARIABLES_PREDICTORAS, VARIABLES_NUMERICAS,
)
from src.preprocessing.particion import (
    dividir_entrenamiento_prueba, guardar_particion, identificar_hogares_panel,
)
from src.preprocessing.preprocesador import crear_preprocesador

# Módulos originales que usa este cuaderno: vivienda (01) y sumaria (34) de 2025, y sumaria de 2024.
# Si ya se descargaron en el cuaderno 01, no se vuelven a descargar.
descargar_enaho(anio=2025, modulos=["01", "34"])
descargar_enaho(anio=2024, modulos=["34"])

hogares = pd.read_csv(RUTA_HOGARES[2025])
print("Dataset de hogares 2025:", hogares.shape)
print("Semilla:", SEMILLA, "| Proporción de prueba:", PROPORCION_PRUEBA)

Dataset de hogares 2025: (33702, 43)
Semilla: 42 | Proporción de prueba: 0.2


## 1. Resumen de las decisiones de preparación

| Transformación | Decisión | Justificación |
|---|---|---|
| Eliminación de duplicados | No se requiere | No hay llaves repetidas ni hogares con contenido idéntico (cuaderno 01) |
| Selección de variables | 36 variables observables | Son características verificables en una visita; se excluyen ingresos, gastos y programas sociales para evitar *leakage* |
| *Feature engineering* | 6 variables derivadas | Resumen información de varias columnas en indicadores con significado socioeconómico (hacinamiento, dependencia, equipamiento, educación) |
| Valores faltantes | Imputación dentro del pipeline | Mediana para numéricas y valor más frecuente para binarias y categóricas, aprendidos solo con entrenamiento |
| Valores atípicos | Se conservan | Son situaciones reales (hogares grandes o hacinados) que describen justamente el fenómeno a detectar |
| Codificación | One-Hot Encoding | Las variables categóricas son códigos nominales; el número del código no representa una cantidad |
| Estandarización | `StandardScaler` en numéricas | La regresión logística y KNN son sensibles a la escala; los árboles no se ven afectados |
| Normalización (Min-Max) | No se aplica | Es más sensible a los valores extremos que la estandarización; con atípicos reales conviene estandarizar |
| Reducción de dimensionalidad | No se aplica | 119 columnas frente a casi 27 000 hogares de entrenamiento es una relación manejable; reducir dimensiones impediría interpretar qué variable influye |
| División de datos | 80 % entrenamiento / 20 % prueba, estratificada | Mantiene la proporción de hogares pobres en ambos conjuntos |
| Validación | Validación cruzada de 5 particiones sobre entrenamiento + validación temporal con la ENAHO 2024 | Estimación robusta y prueba de generalización a otro año |

A continuación se desarrolla y verifica cada decisión.

## 2. Duplicados

In [3]:
print("Llaves de hogar repetidas:", hogares.duplicated(subset=CLAVES_HOGAR).sum())
print("Registros con predictoras y objetivo idénticos:", hogares.duplicated(subset=VARIABLES_PREDICTORAS + [VARIABLE_OBJETIVO]).sum())

Llaves de hogar repetidas: 0
Registros con predictoras y objetivo idénticos: 0


### Interpretación

- **No hay llaves repetidas**: cada fila es un hogar distinto.
- Tampoco hay dos hogares con **las 36 características y la clase idénticas**. Con tantas variables combinadas, cada hogar tiene un perfil único.
- Por lo tanto, **no se elimina ningún registro**.

## 3. Selección de variables

La variable objetivo es `pobre`. Las variables predictoras son las 36 características observables definidas en `src/features/construir.py`. Las columnas de identificación y análisis se separan explícitamente para que **nunca entren al modelo**.

In [4]:
X = hogares[VARIABLES_PREDICTORAS]
y = hogares[VARIABLE_OBJETIVO]

print("Variables predictoras (X):", X.shape[1])
print("Variable objetivo (y):", VARIABLE_OBJETIVO)
print("\nColumnas que NO entran al modelo:", COLUMNAS_IDENTIFICACION)

# Verificación: ninguna columna de identificación ni el objetivo están entre las predictoras
columnas_prohibidas = set(COLUMNAS_IDENTIFICACION + [VARIABLE_OBJETIVO])
print("\n¿Alguna columna prohibida está en X?", bool(columnas_prohibidas & set(X.columns)))

Variables predictoras (X): 36
Variable objetivo (y): pobre

Columnas que NO entran al modelo: ['CONGLOME', 'VIVIENDA', 'HOGAR', 'ubigeo', 'factor07', 'pobreza_inei']

¿Alguna columna prohibida está en X? False


In [5]:
# De las 161 columnas de la sumaria del INEI, ¿cuántas usa el proyecto como predictoras?
columnas_sumaria = leer_csv_enaho(ruta_modulo(2025, "sumaria")).columns
usadas_de_sumaria = ["MIEPERHO (n_miembros)", "DOMINIO (dominio)", "ESTRATO (area_rural)", "UBIGEO (departamento)"]

# Prefijos con los que la sumaria nombra sus columnas de ingresos, gastos, líneas de pobreza y deflactores
prefijos_monetarios = ("ING", "GAS", "GRU", "IA", "IG", "GA", "SG", "SIG", "PAG", "INS", "PAE", "LIN", "LD")
monetarias = [c for c in columnas_sumaria if c.startswith(prefijos_monetarios)]
print("Columnas de la sumaria:", len(columnas_sumaria))
print("Columnas usadas como predictoras:", len(usadas_de_sumaria), "->", usadas_de_sumaria)
print("Columnas monetarias (ingresos, gastos, líneas de pobreza) excluidas:", len(monetarias))
print("Ejemplos de columnas excluidas:", monetarias[:12])

Columnas de la sumaria: 161
Columnas usadas como predictoras: 4 -> ['MIEPERHO (n_miembros)', 'DOMINIO (dominio)', 'ESTRATO (area_rural)', 'UBIGEO (departamento)']
Columnas monetarias (ingresos, gastos, líneas de pobreza) excluidas: 135
Ejemplos de columnas excluidas: ['IA01HD', 'IA02HD', 'IG03HD1', 'IG03HD2', 'IG03HD3', 'IG03HD4', 'GA03HD', 'GA04HD', 'SG23', 'SIG24', 'SG25', 'SIG26']


### Interpretación de la selección de variables

- El modelo recibe **36 variables** y ninguna de las columnas prohibidas.
- De las **161 columnas** de la sumaria, el proyecto solo usa **4 no monetarias**: tamaño del hogar, dominio, estrato (para el área) y ubigeo (para el departamento). Se excluyen **todas** las variables de ingreso, gasto y líneas de pobreza, porque son las que el INEI usa para *calcular* la pobreza. Incluirlas le daría la respuesta al modelo, que es *data leakage* por construcción.
- La selección **no se hizo mirando las métricas**, sino con un criterio previo: usar solo lo que un encuestador puede observar en una visita al hogar.

## 4. Ingeniería de variables (*feature engineering*)

Durante la construcción del dataset (cuaderno 01) se crearon variables derivadas a partir de los módulos de la ENAHO:

| Variable derivada | Cálculo | Justificación |
|---|---|---|
| `personas_por_habitacion` | `n_miembros / n_habitaciones` | Mide el **hacinamiento**, indicador clásico de necesidades básicas insatisfechas |
| `tasa_dependencia` | `(menores de 14 + mayores de 65) / miembros de 14 a 64 años` | Cuántas personas dependen de cada miembro en edad de trabajar |
| `n_menores_14`, `n_mayores_65` | Conteo por edad en el módulo de miembros | Composición demográfica del hogar |
| `jefe_mujer`, `jefe_edad` | Datos del miembro con parentesco "jefe(a)" | Características del responsable del hogar |
| `educacion_jefe`, `max_educacion_adultos` | Nivel educativo (escala 1-11) del jefe(a) y máximo de los adultos | Capital humano del hogar |
| `n_equipos` y 9 variables `tiene_*` | Resumen del módulo de equipamiento (28 artículos) | Riqueza acumulada del hogar |

La siguiente tabla verifica que las variables derivadas tienen un comportamiento distinto según la condición de pobreza:

In [6]:
derivadas = ["personas_por_habitacion", "tasa_dependencia", "n_equipos", "max_educacion_adultos"]
tabla = hogares.groupby(VARIABLE_OBJETIVO)[derivadas].mean().round(2).T
tabla.columns = ["No pobre", "Pobre"]
tabla

,No pobre,Pobre
personas_por_habitacion,1.08,1.78
tasa_dependencia,0.61,0.86
n_equipos,6.25,3.55
max_educacion_adultos,6.83,5.45


### Interpretación

Las cuatro variables derivadas separan a los grupos en la dirección esperada:

- los hogares pobres tienen **más hacinamiento** (1,78 personas por habitación frente a 1,08);
- tienen **mayor dependencia** (0,86 frente a 0,61);
- tienen **menos artefactos** (3,55 frente a 6,25);
- tienen **menor educación máxima de los adultos** (5,45 frente a 6,83).

Las variables tienen sentido conceptual y aportan información, por lo que se conservan.

## 5. División en entrenamiento y prueba

- **80 % entrenamiento / 20 % prueba**, con `train_test_split` (cuaderno 04 del curso).
- **Estratificada** por la variable objetivo, para que ambos conjuntos tengan la misma proporción de hogares pobres (cuaderno 06 del curso).
- Semilla fija (`random_state=42`) para que la división sea reproducible.

El conjunto de prueba **se reserva y no se usa hasta la evaluación final** (cuaderno 05). La selección de hiperparámetros se hará con validación cruzada **dentro del conjunto de entrenamiento**, que actúa como conjunto de validación.

In [7]:
entrenamiento, prueba = dividir_entrenamiento_prueba(hogares)

resumen_division = pd.DataFrame({
    "hogares": [len(entrenamiento), len(prueba)],
    "hogares_pobres": [entrenamiento[VARIABLE_OBJETIVO].sum(), prueba[VARIABLE_OBJETIVO].sum()],
    "porcentaje_pobres": [entrenamiento[VARIABLE_OBJETIVO].mean() * 100, prueba[VARIABLE_OBJETIVO].mean() * 100],
}, index=["Entrenamiento", "Prueba"]).round(2)
resumen_division

,hogares,hogares_pobres,porcentaje_pobres
Entrenamiento,26961,4883,18.11
Prueba,6741,1221,18.11


### Interpretación de la división

- **Entrenamiento:** 26 961 hogares (4 883 pobres).
- **Prueba:** 6 741 hogares (1 221 pobres).
- La proporción de hogares pobres es **idéntica en ambos conjuntos (18,11 %)** gracias a la estratificación. Así, las métricas de prueba no se verán distorsionadas por una distribución de clases distinta.

## 6. Pipeline de preprocesamiento

El preprocesamiento se define con `ColumnTransformer` y `Pipeline` (cuaderno 10 del curso), y aplica un tratamiento distinto según el tipo de variable:

| Grupo | Variables | Paso 1 | Paso 2 |
|---|---|---|---|
| Numéricas | 10 | Imputación con la **mediana** | `StandardScaler` (media 0, desviación 1) |
| Binarias | 16 | Imputación con el **valor más frecuente** | (ya están en escala 0/1) |
| Categóricas | 10 | Imputación con el **valor más frecuente** | `OneHotEncoder(handle_unknown="ignore")` |

**¿Por qué la mediana y no la media?** Porque varias variables numéricas tienen atípicos reales (cuaderno 02), y la mediana no se deja arrastrar por ellos.

**¿Por qué One-Hot y no Label Encoding?** Porque los códigos del INEI son nominales: `material_piso = 6` (tierra) no es "el doble" de `material_piso = 3` (losetas). Con Label Encoding, la regresión logística y KNN interpretarían los códigos como cantidades.

A continuación ajustamos el preprocesador **solo con los datos de entrenamiento**:

In [8]:
preprocesador = crear_preprocesador()
preprocesador.fit(entrenamiento[VARIABLES_PREDICTORAS])

X_entrenamiento_transformado = preprocesador.transform(entrenamiento[VARIABLES_PREDICTORAS])
X_prueba_transformado = preprocesador.transform(prueba[VARIABLES_PREDICTORAS])

print("Entrenamiento transformado:", X_entrenamiento_transformado.shape)
print("Prueba transformada:", X_prueba_transformado.shape)

nombres = pd.Series(preprocesador.get_feature_names_out())
print("\nColumnas generadas por grupo:")
print(nombres.str.split("__").str[0].value_counts())
print("\nValores faltantes después del preprocesamiento:", int(np.isnan(X_entrenamiento_transformado).sum()))

Entrenamiento transformado: (26961, 119)
Prueba transformada: (6741, 119)

Columnas generadas por grupo:
categoricas    93
binarias       16
numericas      10
Name: count, dtype: int64

Valores faltantes después del preprocesamiento: 0


In [9]:
# Lo que el preprocesador "aprendió" del conjunto de entrenamiento
imputador_numerico = preprocesador.named_transformers_["numericas"].named_steps["imputador"]
escalador = preprocesador.named_transformers_["numericas"].named_steps["escalador"]

aprendido = pd.DataFrame({
    "mediana_para_imputar": imputador_numerico.statistics_,
    "media_estandarizacion": escalador.mean_.round(2),
    "desviacion_estandarizacion": escalador.scale_.round(2),
}, index=VARIABLES_NUMERICAS)

print("Hogares con los que se calcularon estos valores:", int(escalador.n_samples_seen_))
aprendido

Hogares con los que se calcularon estos valores: 26961


,mediana_para_imputar,media_estandarizacion,desviacion_estandarizacion
n_miembros,3.0,3.18,1.73
n_habitaciones,3.0,3.27,1.63
personas_por_habitacion,1.0,1.20,0.97
jefe_edad,54.0,54.18,15.73
n_menores_14,0.0,0.68,0.98
n_mayores_65,0.0,0.43,0.68
tasa_dependencia,0.5,0.66,0.68
educacion_jefe,6.0,5.55,2.41
max_educacion_adultos,6.0,6.58,2.50
n_equipos,5.0,5.75,3.56


### Interpretación del pipeline

- Las **36 variables originales se convierten en 119 columnas**: 10 numéricas estandarizadas, 16 binarias y 93 columnas generadas por el One-Hot Encoding de las 10 categóricas. Por ejemplo, `departamento` se convierte en 25 columnas 0/1.
- Después del preprocesamiento **no queda ningún valor faltante**.
- Las medianas, medias y desviaciones se calcularon con **26 961 hogares, exactamente el conjunto de entrenamiento**. El conjunto de prueba solo se *transforma* con esos valores y nunca se usa para calcularlos. Así, el modelo se evalúa como si los hogares de prueba fueran completamente nuevos.
- En los cuadernos 04 y 05, este preprocesador va **dentro del mismo `Pipeline` que el modelo**. Por eso, en cada partición de la validación cruzada se vuelve a ajustar solo con la parte de entrenamiento de esa partición.

## 7. Valores atípicos y reducción de dimensionalidad

**Valores atípicos.** En el cuaderno 02 se identificaron atípicos en personas por habitación (7,0 %) y menores de 14 años (5,7 %). Se **conservan** porque son situaciones reales y relevantes: un hogar hacinado es precisamente un posible hogar pobre. Para limitar su efecto en los modelos sensibles a la escala, se usa la **mediana** en la imputación y la **estandarización** en lugar de Min-Max. Los árboles y Random Forest dividen por umbrales, así que un valor extremo no los afecta.

**Reducción de dimensionalidad.** No se aplica. Con 119 columnas y 26 961 hogares de entrenamiento hay más de 200 observaciones por columna, una relación holgada para los modelos del proyecto. Además, técnicas como PCA combinan las variables en componentes abstractos, lo que impediría explicar qué característica del hogar influye en la predicción. Esa explicación es necesaria para la interpretabilidad y la discusión ética.

## 8. Prevención de data leakage

Esta es una de las secciones más importantes del proyecto. El *data leakage* ocurre cuando el modelo usa, durante el entrenamiento, información que **no tendría disponible en el momento real de la predicción**, lo que produce métricas engañosamente buenas.

### 8.1. ¿Qué información tendría el modelo en el momento real de la predicción?

**Escenario de uso:** un programa social necesita decidir si prioriza a un hogar. Un evaluador lo visita (como se hace con la Ficha Socioeconómica Única) y registra lo que puede **observar o preguntar de forma verificable**. En ese momento **no conoce el gasto ni el ingreso del hogar**: medirlos requeriría una encuesta extensa como la ENAHO, que es justamente lo que se quiere evitar.

| Grupo de información | ¿Disponible en la visita? | ¿Se usa en el modelo? |
|---|---|---|
| Materiales de la vivienda, agua, desagüe, electricidad, combustible | Sí, se observa | **Sí** |
| Artefactos, vehículos y comunicaciones del hogar | Sí, se observa o se pregunta | **Sí** |
| Miembros, edades, sexo y educación | Sí, se pregunta | **Sí** |
| Ubicación (departamento, dominio, área) | Sí, se conoce | **Sí** |
| Ingresos, gastos y líneas de pobreza | No: requieren una encuesta de gasto completa | **No** (son la base de la variable objetivo) |
| Condición de pobreza del INEI (`pobreza_inei`) | No: es lo que se quiere predecir | **No** |
| Programas sociales que recibe el hogar | Es consecuencia de haber sido clasificado como pobre | **No** |
| Factor de expansión (`factor07`) | Es un dato del diseño muestral, no del hogar | **No** |

### 8.2. Verificaciones

In [10]:
verificaciones = {}

# 1. La variable objetivo y sus derivadas no están entre las predictoras
verificaciones["Objetivo y columnas de identificación fuera de X"] = not (set(COLUMNAS_IDENTIFICACION + [VARIABLE_OBJETIVO]) & set(VARIABLES_PREDICTORAS))

# 2. Ningún hogar está a la vez en entrenamiento y en prueba
llaves_entrenamiento = set(map(tuple, entrenamiento[CLAVES_HOGAR].values))
llaves_prueba = set(map(tuple, prueba[CLAVES_HOGAR].values))
verificaciones["Sin hogares compartidos entre entrenamiento y prueba"] = len(llaves_entrenamiento & llaves_prueba) == 0

# 3. El preprocesador se ajustó solo con los hogares de entrenamiento
verificaciones["Preprocesador ajustado solo con entrenamiento"] = int(escalador.n_samples_seen_) == len(entrenamiento)

# 4. Entrenamiento y prueba cubren todo el dataset, sin pérdidas ni repeticiones
verificaciones["Entrenamiento + prueba = dataset completo"] = len(entrenamiento) + len(prueba) == len(hogares)

pd.Series(verificaciones).to_frame("¿se cumple?")

,¿se cumple?
Objetivo y columnas de identificación fuera de X,True
Sin hogares compartidos entre entrenamiento y prueba,True
Preprocesador ajustado solo con entrenamiento,True
Entrenamiento + prueba = dataset completo,True


### 8.3. Hogares del panel en la validación temporal

La ENAHO vuelve a entrevistar cada año a una parte de los hogares (la **muestra panel**). En la validación temporal, el modelo se entrenará con la **ENAHO 2024** y se evaluará con la **ENAHO 2025**. Si un mismo hogar aparece en ambos años, el modelo sería evaluado con familias que ya vio en el entrenamiento, aunque sea un año después. Eso inflaría la estimación de su capacidad de generalizar.

Por eso identificamos a los hogares de 2025 que también fueron entrevistados en 2024 y los **excluimos del conjunto de evaluación temporal**.

In [11]:
panel = identificar_hogares_panel(anio=2025, anio_anterior=2024)
hogares_panel = hogares.merge(panel, on=CLAVES_HOGAR, how="left")

print("Hogares de 2025 que también fueron entrevistados en 2024 (panel):", int(hogares_panel["es_panel"].sum()))
print("Hogares de 2025 nuevos (no panel):", int((hogares_panel["es_panel"] == 0).sum()))
print(f'Porcentaje de hogares pobres entre los nuevos: {hogares_panel.loc[hogares_panel["es_panel"] == 0, VARIABLE_OBJETIVO].mean() * 100:.2f} %')

Hogares de 2025 que también fueron entrevistados en 2024 (panel): 9842
Hogares de 2025 nuevos (no panel): 23860
Porcentaje de hogares pobres entre los nuevos: 17.65 %


### Interpretación

- **9 842 hogares** de 2025 (29 %) también fueron entrevistados en 2024. Quedan **excluidos** de la evaluación temporal.
- La validación temporal se hará con los **23 860 hogares nuevos** de 2025. Su proporción de pobres (17,65 %) es similar a la del total, así que el subconjunto sigue siendo representativo.
- Este tipo de fuga es fácil de pasar por alto, porque los hogares repetidos tienen las **mismas llaves** en ambos años.

### 8.4. Una limitación que se documenta: hogares vecinos del mismo conglomerado

La ENAHO selecciona **conglomerados** (grupos de viviendas vecinas) y luego hogares dentro de ellos. Con una división aleatoria por hogar, dos vecinos del mismo conglomerado pueden quedar uno en entrenamiento y otro en prueba. Los vecinos comparten condiciones del entorno, lo que podría hacer las métricas algo optimistas para zonas que el modelo nunca vio. **No es información de la variable objetivo**, así que no es *leakage* en sentido estricto. Su efecto se mide en la sección 10, después de guardar la partición, y se discute en las **amenazas a la validez** del informe.

### 8.5. Resumen de las medidas contra el data leakage

| Riesgo | Medida aplicada |
|---|---|
| Variables que contienen la respuesta (ingreso, gasto, líneas de pobreza) | Excluidas por diseño; solo características observables |
| Variables que son consecuencia de la pobreza (programas sociales) | Excluidas |
| Calcular imputación y escalado con datos de prueba | Preprocesador dentro de un `Pipeline`, ajustado solo con entrenamiento |
| Validación cruzada contaminada | El `Pipeline` completo se reajusta en cada partición |
| Mismo hogar en entrenamiento y prueba | Verificado: 0 hogares compartidos |
| Mismo hogar en 2024 (entrenamiento) y 2025 (evaluación temporal) | Se excluyen los 9 842 hogares del panel |
| Usar el conjunto de prueba para decidir | La prueba se usa una sola vez, en la evaluación final (cuaderno 05) |

## 9. Guardado de los artefactos

Guardamos la **partición** (qué hogar está en entrenamiento y cuál en prueba) y la lista de **hogares nuevos de 2025** (no panel). Así, los cuadernos 04 y 05 usan exactamente los mismos conjuntos. El preprocesador no se guarda aquí: se define en `src/preprocessing/preprocesador.py` y se ajusta junto con cada modelo.

In [12]:
guardar_particion(entrenamiento, prueba, RUTA_PARTICION)
print("Partición guardada en:", RUTA_PARTICION)

panel.to_csv(RUTA_PANEL_2025, index=False)
print("Marca de hogares del panel guardada en:", RUTA_PANEL_2025)

Partición guardada en: data/processed/particion_2025.csv
Marca de hogares del panel guardada en: data/processed/panel_2025.csv


## 10. Hogares vecinos en la validación cruzada

En la sección 8.4 vimos que la división por hogar deja vecinos del mismo conglomerado en entrenamiento y en prueba. ¿Cuánto podría inflar eso las métricas? Podemos medirlo **sin otro experimento**, aprovechando cómo funciona la validación cruzada del cuaderno 04.

Con `cv=5`, `GridSearchCV` usa `StratifiedKFold` **sin barajar**: cada partición es un bloque contiguo de filas, en el orden en que `aplicar_particion` entrega el entrenamiento. Como los hogares de un mismo conglomerado están seguidos en el dataset, casi todos los conglomerados deberían quedar enteros dentro de una sola partición. Lo comprobamos reproduciendo esas mismas cinco particiones.

In [13]:
from sklearn.model_selection import StratifiedKFold
from src.config import FOLDS_VALIDACION_CRUZADA
from src.preprocessing.particion import aplicar_particion

# Mismo entrenamiento y mismo orden de filas que usa el cuaderno 04
entrenamiento_cv, prueba_cv = aplicar_particion(hogares, RUTA_PARTICION)

# Reproducimos las 5 particiones de cv=5 (StratifiedKFold sin barajar) y anotamos la de cada hogar
particion_cv = pd.Series(-1, index=entrenamiento_cv.index)
divisor = StratifiedKFold(n_splits=FOLDS_VALIDACION_CRUZADA)
for numero, (_, indices) in enumerate(divisor.split(entrenamiento_cv, entrenamiento_cv[VARIABLE_OBJETIVO])):
    particion_cv.iloc[indices] = numero

# ¿En cuántas particiones distintas cae cada conglomerado? (1 = queda entero en una sola)
particiones_por_conglomerado = particion_cv.groupby(entrenamiento_cv["CONGLOME"]).nunique()
print("Conglomerados en el entrenamiento:", len(particiones_por_conglomerado))
print(f"Conglomerados enteros dentro de una sola partición: {(particiones_por_conglomerado == 1).mean() * 100:.1f} %")
print("Departamentos presentes en cada partición:",
      pd.crosstab(entrenamiento_cv["departamento"], particion_cv).astype(bool).sum().tolist())

# En la prueba, en cambio: ¿cuántos hogares tienen vecinos de su conglomerado en el entrenamiento?
con_vecinos = prueba_cv["CONGLOME"].isin(set(entrenamiento_cv["CONGLOME"]))
print(f"Hogares de prueba con vecinos del mismo conglomerado en el entrenamiento: {con_vecinos.mean() * 100:.1f} %")

Conglomerados en el entrenamiento: 5358
Conglomerados enteros dentro de una sola partición: 98.5 %
Departamentos presentes en cada partición: [25, 25, 25, 25, 25]
Hogares de prueba con vecinos del mismo conglomerado en el entrenamiento: 99.9 %


### Interpretación

- El **98,5 % de los conglomerados** queda entero dentro de una sola partición. En cada paso de la validación cruzada, el modelo se evalúa con hogares cuyos **vecinos no vio al entrenar**.
- En cambio, el **99,9 % de los hogares de prueba** tiene vecinos en el entrenamiento.
- Las cinco particiones incluyen los 25 departamentos: la validación cruzada separa a los vecinos, pero **no evalúa regiones completas no vistas**.

Esto da una forma sencilla de medir el efecto de los vecinos: si inflaran las métricas, el F1 en la prueba (con vecinos) sería claramente mayor que en la validación cruzada (sin vecinos). La comparación se hace en el cuaderno 05, sección 5, cuando ya se conocen ambos resultados.

## Conclusiones del cuaderno

1. **No fue necesario eliminar registros**: no hay duplicados, y los atípicos se conservan porque describen situaciones reales de pobreza y hacinamiento.
2. Se seleccionaron **36 variables observables** con un criterio previo, no por métricas. Se excluyeron todas las variables monetarias de la sumaria y los programas sociales.
3. Se incorporaron **variables derivadas** con significado socioeconómico (hacinamiento, dependencia, equipamiento, educación) que diferencian a hogares pobres y no pobres.
4. Los datos se dividieron en **entrenamiento (26 961) y prueba (6 741)** de forma estratificada, con la misma proporción de pobres (18,11 %).
5. El **pipeline de preprocesamiento** imputa, codifica y estandariza según el tipo de variable, generando 119 columnas sin faltantes. Todo lo que aprende proviene solo del entrenamiento.
6. Se **demostró la ausencia de data leakage** con verificaciones explícitas y se preparó una validación temporal libre de hogares repetidos (23 860 hogares nuevos de 2025).
7. La validación cruzada deja el **98,5 % de los conglomerados** enteros en una sola partición, así que evalúa con vecinos no vistos. Esto permitirá medir en el cuaderno 05 si los vecinos inflan las métricas de la prueba.

**Siguiente paso:** cuaderno `04_modeling.ipynb`, con el baseline, los cuatro modelos, la búsqueda de hiperparámetros y el diseño experimental.